# Simulación controlada SIR–SI
Este cuaderno presenta una réplica con tasas verdaderas conocidas. El estudio completo se ejecutó con `simulacion.py` y sus resultados están en los CSV e informe.

Se observan **casos semanales**, no estados infectados. Para presentar: explicar el modelo, generar datos, ajustar, comparar parámetros y reintegrar.

El archivo modelo_para_exponer.py contiene el paso Euler–Maruyama en NumPy, verificado contra el motor acelerado.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import metodos as m
m.OUT = ROOT / 'results' / 'models'
m.OUT.mkdir(parents=True, exist_ok=True)
from simulacion import generar, VERDAD, X0, SEMANAS


## 1. Verdad y datos
Verdad artificial: βh=.35, βv=.45, γ=.10 por día. Nh=4 658 811 proviene del DANE para Valle del Cauca en 2022. ρ=.20, μv=.10, σ y X0 son supuestos conocidos del experimento.
Euler–Maruyama genera una trayectoria a paso .025 días. Cada semana acumula ρNhβhShIv dt. No se añade error de observación.


In [ ]:
sigma = (.12, .15)
m.C.update(sigma_h=sigma[0], sigma_v=sigma[1])
datos, intensidad, estados_verdaderos = generar(730001, sigma)
plt.plot(np.arange(1, SEMANAS+1), datos.casos, 'o-', label='Casos sintéticos')
plt.xlabel('Semana'); plt.ylabel('Casos'); plt.legend(); plt.show()


## 2. Los tres métodos
TRF y SGD minimizan la misma pérdida de incidencia media de 32 trayectorias independientes. PINN aproxima estados latentes y añade una penalización de transición estocástica. Sus objetivos completos difieren.

SGD: 26 observaciones, lotes de 13 → dos lotes por época; hasta 4000 actualizaciones.
PINN: dos capas de 32 tanh, Adam 2000 pasos y L-BFGS hasta 2000 iteraciones.


In [ ]:
inicio = [.15, .20, 1/7]
ajustes = {}
for metodo in ['TRF', 'SGD']:
    ajustes[metodo] = m.fit_direct(metodo, datos, X0, inicio, seed=810011)
ajustes['PINN'] = m.fit_pinn(datos, X0, inicio, 810011,
                            'demostracion', np.arange(SEMANAS), SEMANAS)


## 3. Recuperación de tasas y ajuste
Comparar cada tasa con su verdad. Una curva visualmente cercana no garantiza parámetros correctos. Mantener visibles las interrupciones y las causas de parada.


In [ ]:
tabla = pd.DataFrame([dict(metodo=nombre, **salida[0])
                      for nombre, salida in ajustes.items()])
display(tabla[['metodo','beta_h','beta_v','gamma','estado','terminacion']])
print('Verdad:', dict(zip(['beta_h','beta_v','gamma'], VERDAD)))
for nombre, (info,pred,x,hist) in ajustes.items():
    if pred is not None:
        plt.plot(np.arange(1,SEMANAS+1), pred, label=nombre)
plt.scatter(np.arange(1,SEMANAS+1), datos.casos, c='black', label='Datos')
plt.xlabel('Semana'); plt.ylabel('Casos'); plt.legend(); plt.show()


## 4. Reintegración y coherencia física
Reintegrar tasas PINN en EM permite evaluar si describen la curva que aprendió la red. Comprobar además Rh no decreciente, pues dRh=γIh dt≥0.


In [ ]:
info, pred, x, hist = ajustes['PINN']
if pred is not None:
    tasas = np.array([info[k] for k in ['beta_h','beta_v','gamma']])
    try:
        em, _, _ = m.Simulator(datos,X0)(np.log(tasas))
        print('RMSE red:', np.sqrt(np.mean((pred-datos.casos)**2)))
        print('RMSE reintegrado:', np.sqrt(np.mean((em-datos.casos)**2)))
    except FloatingPointError as error:
        print('Reintegración fuera del dominio:', error)
    print('Mínimo incremento de Rh:', np.diff(x[:,2]).min())


## 5. Estudio completo
Ejecutar `python src/simulacion.py --replicas 10` desde la raíz. Dos niveles de ruido del proceso, diez réplicas y dos inicios por método. Los datos reales se tratarán posteriormente como una trayectoria continua de 156 semanas.
